# Phase 17 — Experimental modelling framework (stage 31)

| | |
| --- | --- |
| **Workflow** | `final workflow.md` **Phase 17 — Experimental modelling framework** → checklist **stage 31** |
| **Built by** | `scripts/phase17_framework.py` → `scripts/phase17_run.log` (exit 0, 20/20) |
| **This notebook** | independent re-verification of the written registry from the frozen inputs |
| **Inputs** | `reports/phase17_experiment_registry.csv` · `split_assignments.csv` · `feature_sets.json` · `phase16_baseline_metrics.csv` · `experimental_dataset.csv` (SHA re-checked) |
| **Output** | `reports/phase17_framework.md` + registry CSV (unchanged by this notebook) |
| **Binding decisions** | **D2** three separate horizons · **D11** numbering · **D12** L ∈ {3,7,14,30} empirical · **D13** L4 gated · **D14** ML-vs-DL parity · **D7** persistence bar |

**Fourteen checks, N17-01 … N17-14.** No model is trained here and no test cell is
read for any decision — the frozen 2023+ cells are only *constructed* to prove they
exist for Phase 33.

In [1]:
import hashlib, json, time, warnings
from pathlib import Path

import numpy as np
import pandas as pd

warnings.filterwarnings("ignore")

ROOT = Path(r"D:\Flood predictor")
PROC = ROOT / "data" / "processed"
REPORTS = ROOT / "reports"
TBL = PROC / "experimental_dataset.csv"
SPLITS = PROC / "split_assignments.csv"
FSETS = PROC / "feature_sets.json"
REG = REPORTS / "phase17_experiment_registry.csv"
REPORT_MD = REPORTS / "phase17_framework.md"
P16_MET = REPORTS / "phase16_baseline_metrics.csv"

TBL_SHA_EXPECTED = "a6748529083fcdceb8bb8a657b067bf12b5c861f3a7a6b09c8977ea63be73c8e"
SEED = 20261008
HORIZONS = ["label_24h", "label_48h", "label_72h"]
SEASONS = ["all", "monsoon", "non-monsoon"]
ROLLING_YEARS = [2020, 2021, 2022]
SPATIAL_FOLDS = [1, 2, 3, 4]
TEST_YEARS = [2023, 2024, 2025, 2026]
TEST_BUCKETS = ["test_trained", "test_new_network", "test_unseen"]
HOLDOUT_CUTOFF = 2023
L1_MODELS = ["climatology", "persistence", "logreg_linear", "logreg_linear_no_up",
             "logreg_selected", "tree_depth4"]


def sha256(p):
    h = hashlib.sha256()
    with open(p, "rb") as f:
        for blk in iter(lambda: f.read(1 << 20), b""):
            h.update(blk)
    return h.hexdigest()


RES, WARN = [], []


def check(cid, detail, ok=True):
    RES.append({"id": cid, "pass": bool(ok)})
    print(f"[{'PASS' if ok else 'FAIL'}] {cid}: {detail}")
    assert ok, cid


def warn(wid, detail):
    WARN.append({"id": wid, "note": detail})
    print(f"[WARN] {wid}: {detail}")


t0 = time.time()
print("Phase 17 notebook - registry re-verification")

Phase 17 notebook - registry re-verification


In [2]:
# N17-01 - inputs present, source table frozen
check("N17-01a", f"registry CSV present ({REG.stat().st_size:,} bytes) and "
                 f"framework report present ({REPORT_MD.stat().st_size:,} bytes)",
      REG.exists() and REPORT_MD.exists())
sha = sha256(TBL)
check("N17-01b", f"experimental_dataset.csv SHA-256 unchanged ({sha[:16]}...)",
      sha == TBL_SHA_EXPECTED)
s = pd.read_csv(SPLITS)
fs = json.loads(FSETS.read_text(encoding="utf-8"))
check("N17-01c", f"split table {s.shape[0]:,} x {s.shape[1]}; feature sets "
                 f"{fs['counts']}; banned columns {len(fs['banned'])} listed",
      s.shape == (200665, 9) and set(fs["sets"]) == {"FULL", "LINEAR", "SELECTED"})

[PASS] N17-01a: registry CSV present (10,552 bytes) and framework report present (9,918 bytes)


[PASS] N17-01b: experimental_dataset.csv SHA-256 unchanged (a6748529083fcdce...)


[PASS] N17-01c: split table 200,665 x 9; feature sets {'FULL': 62, 'LINEAR': 35, 'SELECTED': 10}; banned columns 10 listed


In [3]:
# N17-02 - registry structure
reg = pd.read_csv(REG)
need = {"registry_id", "level", "family", "model", "dataset", "feature_set",
        "reason", "dev_cells", "seasons", "horizons", "selection_metric",
        "selection_cell", "seed", "tuning_budget_configs", "imbalance_policy",
        "status", "gate", "test_policy"}
check("N17-02a", f"registry {reg.shape[0]} experiments x {reg.shape[1]} columns; "
                 f"all contract columns present",
      need <= set(reg.columns))
counts = reg.level.value_counts().to_dict()
check("N17-02b", f"four levels with L1=6 (done) L2=5 L3=9 (6 + 3 Phase-23 fusion "
                 f"amendments) L4=4: {counts}",
      counts == {"L1": 6, "L2": 5, "L3": 9, "L4": 4})
check("N17-02c", f"registry ids unique ({reg.registry_id.nunique()}) and model names "
                 f"unique ({reg.model.nunique()})",
      reg.registry_id.is_unique and reg.model.is_unique)
print(reg.groupby(["level", "family"]).size().to_string())

[PASS] N17-02a: registry 24 experiments x 22 columns; all contract columns present
[PASS] N17-02b: four levels with L1=6 (done) L2=5 L3=9 (6 + 3 Phase-23 fusion amendments) L4=4: {'L3': 9, 'L1': 6, 'L2': 5, 'L4': 4}
[PASS] N17-02c: registry ids unique (24) and model names unique (24)
level  family                 
L1     simple statistical         6
L2     classical ML               5
L3     deep temporal              6
       feature fusion             3
L4     advanced spatiotemporal    4


In [4]:
# N17-03 - every candidate carries a reason (anti-padding rule)
lens = reg.reason.str.len()
check("N17-03", f"all {len(reg)} candidates have a written reason "
                f"(min {lens.min()} chars, median {int(lens.median())}); "
                f"no empty/generic justification",
      lens.min() >= 40 and reg.reason.notna().all())
print(reg[["registry_id", "level", "model", "reason"]].head(6).to_string(index=False))

[PASS] N17-03: all 24 candidates have a written reason (min 74 chars, median 105); no empty/generic justification
registry_id level               model                                                                                                                reason
      L1-01    L1         climatology                    constant train base rate - the prevalence reference; PR-AUC of any model must be judged against it
      L1-02    L1         persistence D7 bar (D7): one feature, monotone logistic link, beats every simple baseline - the benchmark all families must clear
      L1-03    L1       logreg_linear               regularised linear model on the Phase-12 LINEAR set - the honest linear ceiling for engineered features
      L1-04    L1 logreg_linear_no_up                                            F14-2 ablation: does dropping the 67-70 % NaN up_* columns change anything
      L1-05    L1     logreg_selected                                            compact 10-feature cross-

In [5]:
# N17-04 - L1 == the six Phase-16 baselines, already run, test read once
p16 = pd.read_csv(P16_MET)
l1 = reg[reg.level == "L1"]
check("N17-04a", f"L1 model set == Phase-16 metric file's six models "
                 f"{sorted(l1.model.unique())}",
      set(l1.model) == set(p16.model.unique()) == set(L1_MODELS))
check("N17-04b", "L1 status=done with test_policy=read_once_phase16 (their single "
                 "frozen-test read already happened in Phase 16)",
      (l1.status == "done").all()
      and (l1.test_policy == "read_once_phase16").all())
check("N17-04c", f"Phase-16 metrics file intact: {len(p16)} rows = 8 tasks x 6 models "
                 f"x 3 horizons x 3 seasons",
      len(p16) == 432)

[PASS] N17-04a: L1 model set == Phase-16 metric file's six models ['climatology', 'logreg_linear', 'logreg_linear_no_up', 'logreg_selected', 'persistence', 'tree_depth4']
[PASS] N17-04b: L1 status=done with test_policy=read_once_phase16 (their single frozen-test read already happened in Phase 16)
[PASS] N17-04c: Phase-16 metrics file intact: 432 rows = 8 tasks x 6 models x 3 horizons x 3 seasons


In [6]:
# N17-05 - fair-comparison parity (D14): planned runs are identical on every axis
plan = reg[reg.level != "L1"]
cols = ["protocols", "dev_cells", "seasons", "horizons", "selection_metric",
        "selection_cell", "seed", "test_policy"]
uniform = {c: plan[c].nunique() == 1 for c in cols}
check("N17-05a", f"parity columns single-valued across all planned runs: {uniform}",
      all(uniform.values()))
check("N17-05b", f"shared values: protocols={plan.protocols.iloc[0]}, "
                 f"seasons={plan.seasons.iloc[0]}, horizons={plan.horizons.iloc[0]}, "
                 f"selection={plan.selection_metric.iloc[0]} on "
                 f"{plan.selection_cell.iloc[0]}, seed={plan.seed.iloc[0]}",
      plan.protocols.iloc[0] == "P1,P2,P3"
      and plan.seasons.iloc[0] == "all,monsoon,non-monsoon"
      and plan.horizons.iloc[0] == "24h,48h,72h"
      and plan.selection_metric.iloc[0] == "pr_auc"
      and plan.selection_cell.iloc[0] == "val_2022"
      and plan.seed.iloc[0] == SEED)
check("N17-05c", "imbalance policy declared inside-training only for every planned run "
                 f"({sorted(plan.imbalance_policy.unique())}) - never resampled "
                 "before the split (R4)",
      set(plan.imbalance_policy) <= {"class_weight=balanced", "none"})

[PASS] N17-05a: parity columns single-valued across all planned runs: {'protocols': True, 'dev_cells': True, 'seasons': True, 'horizons': True, 'selection_metric': True, 'selection_cell': True, 'seed': True, 'test_policy': True}
[PASS] N17-05b: shared values: protocols=P1,P2,P3, seasons=all,monsoon,non-monsoon, horizons=24h,48h,72h, selection=pr_auc on val_2022, seed=20261008
[PASS] N17-05c: imbalance policy declared inside-training only for every planned run (['class_weight=balanced']) - never resampled before the split (R4)


In [7]:
# N17-06 - test protection: no planned run sees a frozen test cell
bad = plan[plan.dev_cells.str.contains("test")
           | plan.selection_cell.str.startswith("test")]
check("N17-06a", f"no planned run has a test cell in its development grid or selection "
                 f"cell ({len(bad)} violations)",
      bad.empty and (plan.test_policy == "frozen_read_at_phase33").all())

# construct the seven frozen cells straight from the Phase-15 masks
split, year = s.split.to_numpy(), pd.to_datetime(s.date).dt.year.to_numpy()
fold, hold, fy = s.spatial_fold.to_numpy(), s.station_holdout.to_numpy(), s.first_year.to_numpy()
cells_frozen = ([f"test_{y}" for y in TEST_YEARS if ((split == "test") & (year == y)).any()]
                + [b for b, m in [("test_trained", (split == "test") & ~hold & (fy <= HOLDOUT_CUTOFF)),
                                  ("test_new_network", (split == "test") & ~hold & (fy > HOLDOUT_CUTOFF)),
                                  ("test_unseen", (split == "test") & hold)] if m.any()])
check("N17-06b", f"the seven frozen test cells exist for the Phase-33 read: "
                 f"{cells_frozen}",
      cells_frozen == [f"test_{y}" for y in TEST_YEARS] + TEST_BUCKETS)

[PASS] N17-06a: no planned run has a test cell in its development grid or selection cell (0 violations)


[PASS] N17-06b: the seven frozen test cells exist for the Phase-33 read: ['test_2023', 'test_2024', 'test_2025', 'test_2026', 'test_trained', 'test_new_network', 'test_unseen']


In [8]:
# N17-07 - tuning budgets fixed before any result exists
budgets = {lvl: sorted(g.tuning_budget_configs.unique().tolist())
           for lvl, g in reg.groupby("level")}
check("N17-07", f"budgets per level: {budgets} (L1=0 reference, L2=40, L3 temporal=20 / "
                "L3 fusion=16 (Phase-23 amendment, within the L3 cap), L4=20 - cannot "
                "expand after a model looks promising)",
      budgets == {"L1": [0], "L2": [40], "L3": [16, 20], "L4": [20]})

[PASS] N17-07: budgets per level: {'L1': [0], 'L2': [40], 'L3': [16, 20], 'L4': [20]} (L1=0 reference, L2=40, L3 temporal=20 / L3 fusion=16 (Phase-23 amendment, within the L3 cap), L4=20 - cannot expand after a model looks promising)


In [9]:
# N17-08 - L4 gates (D13): Phase-24 and Phase-25 gates both discharged
l4 = reg[reg.level == "L4"]
p24 = l4[l4.gate == "phase24_graph_gate"]
p25 = l4[l4.gate == "phase25_transformer_gate"]
check("N17-08a", f"{len(p24)} L4 rows discharged status=planned (phase24_graph_gate "
                 f"passed), {len(p25)} discharged status=planned "
                 f"(phase25_transformer_gate passed)",
      len(l4) == 4 and set(p24.registry_id) == {"L4-01", "L4-02", "L4-04"}
      and (p24.status == "planned").all()
      and set(p25.registry_id) == {"L4-03"} and (p25.status == "planned").all()
      and set(l4.gate) == {"phase24_graph_gate", "phase25_transformer_gate"})
check("N17-08b", "no other level carries a gate; nothing outside L4 claims to be "
                 "blocked",
      (reg[reg.level != "L4"].gate.fillna("") == "").all()
      and set(reg[reg.level != "L4"].status) == {"done", "planned"})
print(l4[["registry_id", "model", "gate", "status"]].to_string(index=False))

[PASS] N17-08a: 3 L4 rows discharged status=planned (phase24_graph_gate passed), 1 discharged status=planned (phase25_transformer_gate passed)
[PASS] N17-08b: no other level carries a gate; nothing outside L4 claims to be blocked
registry_id                model                     gate  status
      L4-01                  gcn       phase24_graph_gate planned
      L4-02                  gat       phase24_graph_gate planned
      L4-03 transformer_temporal phase25_transformer_gate planned
      L4-04             gnn_lstm       phase24_graph_gate planned


In [10]:
# N17-09 - feature sets resolve to Phase-12 sets / raw-sequence spec
tab = reg[reg.dataset == "A"]
known = set(fs["sets"]) | {"none", "wl_pct_danger", "LINEAR-no-up"}
check("N17-09a", f"Dataset-A feature sets all known: {sorted(tab.feature_set.unique())}",
      set(tab.feature_set) <= known)
seq = reg[reg.dataset == "B"]
lengths = set()
for spec in seq.feature_set:
    if "L=" in spec:
        lengths.add(spec.split("L=")[1].split("|")[0])
check("N17-09b", f"Dataset-B rows are raw_sequence(12 vars) with L in "
                 f"{{3,7,14,30}}/best/graph: {sorted(lengths)}",
      seq.feature_set.str.startswith("raw_sequence(12 vars)").all()
      and lengths <= {"3", "7", "14", "30", "best"}
      and any("graph" in x for x in seq.feature_set))
check("N17-09c", "every feature set count matches the Phase-12 manifest "
                 f"{fs['counts']}",
      fs["counts"]["FULL"] == 62 and fs["counts"]["LINEAR"] == 35
      and fs["counts"]["SELECTED"] == 10)

[PASS] N17-09a: Dataset-A feature sets all known: ['FULL', 'LINEAR', 'LINEAR-no-up', 'SELECTED', 'none', 'wl_pct_danger']
[PASS] N17-09b: Dataset-B rows are raw_sequence(12 vars) with L in {3,7,14,30}/best/graph: ['14', '3', '30', '7', 'best']
[PASS] N17-09c: every feature set count matches the Phase-12 manifest {'FULL': 62, 'LINEAR': 35, 'SELECTED': 10}


In [11]:
# N17-10 - grid arithmetic: the cells every planned model will be scored on
p1 = ["val_2022"] if ((s.split == "val") & ~s.station_holdout).any() else []
p2 = [f"val_{y}" for y in ROLLING_YEARS if ((year == y) & ~s.station_holdout).any()]
p3 = [f"spatial_fold_{k}" for k in SPATIAL_FOLDS
      if ((s.split == "train") & ~s.station_holdout & (fold == k)).any()]
n_tasks = len(p1) + len(p2) + len(p3)
cells_model = n_tasks * len(SEASONS) * len(HORIZONS)
check("N17-10a", f"development tasks supported by the masks: P1={p1}, P2={p2}, "
                 f"P3={p3} -> {n_tasks} tasks",
      n_tasks == 8 and p1 == ["val_2022"]
      and p2 == [f"val_{y}" for y in ROLLING_YEARS]
      and p3 == [f"spatial_fold_{k}" for k in SPATIAL_FOLDS])
check("N17-10b", f"{n_tasks} tasks x {len(SEASONS)} seasons x {len(HORIZONS)} horizons "
                 f"= {cells_model} cells per model = the Phase-16 grid "
                 f"({len(p16)} rows / 6 models)",
      cells_model == 72 and len(p16) / 6 == cells_model)

[PASS] N17-10a: development tasks supported by the masks: P1=['val_2022'], P2=['val_2020', 'val_2021', 'val_2022'], P3=['spatial_fold_1', 'spatial_fold_2', 'spatial_fold_3', 'spatial_fold_4'] -> 8 tasks
[PASS] N17-10b: 8 tasks x 3 seasons x 3 horizons = 72 cells per model = the Phase-16 grid (432 rows / 6 models)


In [12]:
# N17-11 - three separate tasks (D2) and the D7 bar (D7)
check("N17-11a", "every experiment declares all three horizon tasks separately "
                 f"({set(reg.horizons)}) - no pooled horizon anywhere",
      set(reg.horizons) == {"24h,48h,72h"})
check("N17-11b", "D7 persistence benchmark present as L1-02 - every family is reported "
                 "against it",
      (reg[reg.registry_id == "L1-02"].model == "persistence").all())
check("N17-11c", "D12 window lengths present as explicit LSTM rows "
                 "(L3-01..04 = L 3/7/14/30)",
      set(reg[reg.model.str.startswith("lstm_L")].model)
      == {"lstm_L3", "lstm_L7", "lstm_L14", "lstm_L30"})

[PASS] N17-11a: every experiment declares all three horizon tasks separately ({'24h,48h,72h'}) - no pooled horizon anywhere
[PASS] N17-11b: D7 persistence benchmark present as L1-02 - every family is reported against it
[PASS] N17-11c: D12 window lengths present as explicit LSTM rows (L3-01..04 = L 3/7/14/30)


In [13]:
# N17-12 - report and registry agree
txt = REPORT_MD.read_text(encoding="utf-8")
ids_in_report = [i for i in reg.registry_id if i in txt]
check("N17-12a", f"report quotes {len(ids_in_report)}/{len(reg)} registry ids "
                 "(or their rows) and names the registry file",
      "phase17_experiment_registry.csv" in txt and len(ids_in_report) >= 8)
check("N17-12b", "report states the gate rule (D13) and the parity rule (D14) that the "
                 "CSV encodes",
      "gated" in txt.lower() and "parity" in txt.lower() and "D14" in txt)
check("N17-12c", "report records the test-protection rule the CSV encodes",
      "frozen_read_at_phase33" in txt or "Phase 33" in txt)

[PASS] N17-12a: report quotes 24/24 registry ids (or their rows) and names the registry file
[PASS] N17-12b: report states the gate rule (D13) and the parity rule (D14) that the CSV encodes
[PASS] N17-12c: report records the test-protection rule the CSV encodes


In [14]:
print("=" * 78)
print("PHASE 17 / STAGE 31   experiment framework verified - 24 runs registered")
print("=" * 78)
for r in RES:
    print(f"  {'PASS' if r['pass'] else 'FAIL'}  {r['id']}")
for w in WARN:
    print(f"  WARN  {w['id']}: {w['note']}")
print(f"\n{sum(r['pass'] for r in RES)}/{len(RES)} checks passed, {len(WARN)} warnings, "
      f"{time.time() - t0:.1f}s")
print("registry levels:", reg.groupby("level").size().to_dict())
print("next: Phase 18 - ML/DL dataset preparation (Dataset B, L in {3,7,14,30})")
assert all(r["pass"] for r in RES)
assert sha256(TBL) == TBL_SHA_EXPECTED, "experimental table changed!"
print("ALL PHASE-17 CHECKS PASSED - experimental_dataset.csv byte-stable")

PHASE 17 / STAGE 31   experiment framework verified - 24 runs registered
  PASS  N17-01a
  PASS  N17-01b
  PASS  N17-01c
  PASS  N17-02a
  PASS  N17-02b
  PASS  N17-02c
  PASS  N17-03
  PASS  N17-04a
  PASS  N17-04b
  PASS  N17-04c
  PASS  N17-05a
  PASS  N17-05b
  PASS  N17-05c
  PASS  N17-06a
  PASS  N17-06b
  PASS  N17-07
  PASS  N17-08a
  PASS  N17-08b
  PASS  N17-09a
  PASS  N17-09b
  PASS  N17-09c
  PASS  N17-10a
  PASS  N17-10b
  PASS  N17-11a
  PASS  N17-11b
  PASS  N17-11c
  PASS  N17-12a
  PASS  N17-12b
  PASS  N17-12c

29/29 checks passed, 0 warnings, 1.6s
registry levels: {'L1': 6, 'L2': 5, 'L3': 9, 'L4': 4}
next: Phase 18 - ML/DL dataset preparation (Dataset B, L in {3,7,14,30})


ALL PHASE-17 CHECKS PASSED - experimental_dataset.csv byte-stable
